# Trace Analytics

Read-only analytics for wrapper-stored recordings and V2 agent traces in `__data1/`.

This notebook expects the current flat schemas:

- `__data1/recordings.json` with top-level `records`
- `__data1/agent-traces.json` with top-level `runs`

It does not import `app.py`, start Flask, call the LLM, or write to `__data1/`.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd

LEGACY_TICKS_PER_SECOND = 16

# If you open the notebook from another working directory, set this manually.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "__data1").exists() and (REPO_ROOT.parent / "__data1").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "__data1"
RECORDINGS_PATH = DATA_DIR / "recordings.json"
TRACES_PATH = DATA_DIR / "agent-traces.json"

print(f"repo root: {REPO_ROOT}")
print(f"data dir:  {DATA_DIR}")


## Load Stores

Missing files or empty stores produce empty dataframes so the notebook can still run in a fresh checkout.


In [ ]:
def load_json(path: Path, default: dict[str, Any]) -> dict[str, Any]:
    if not path.exists():
        print(f"missing: {path}")
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"invalid JSON in {path}: {exc}") from exc

recording_store = load_json(RECORDINGS_PATH, {"version": 1, "records": {}})
trace_store = load_json(TRACES_PATH, {"version": 1, "runs": {}})

records = recording_store.get("records") or {}
runs = trace_store.get("runs") or {}

print(f"records: {len(records)}")
print(f"runs:    {len(runs)}")


## Normalize JSON To Tables

The table builders intentionally read only the current V2 schema. They avoid legacy migration logic.


In [ ]:
def iso_to_datetime(value: Any) -> pd.Timestamp:
    if not value:
        return pd.NaT
    return pd.to_datetime(value, utc=True, errors="coerce")


def list_len(value: Any) -> int:
    return len(value) if isinstance(value, list) else 0


def nested_get(data: dict[str, Any], *keys: str, default: Any = None) -> Any:
    current: Any = data
    for key in keys:
        if not isinstance(current, dict):
            return default
        current = current.get(key)
    return default if current is None else current


def build_records_df(records: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record_id, record in records.items():
        demo = record.get("demo") or {}
        solver = record.get("solver") or {}
        action_count = list_len(demo.get("action")) // 2
        demo_time_ticks = pd.to_numeric(demo.get("time"), errors="coerce")
        rows.append({
            "recordId": record.get("id") or record_id,
            "traceId": record.get("traceId"),
            "source": record.get("source"),
            "result": record.get("result"),
            "playData": record.get("playData"),
            "level": record.get("level"),
            "savedAt": iso_to_datetime(record.get("savedAt")),
            "demoState": demo.get("state"),
            "demoTimeTicks": demo_time_ticks,
            "demoTimeSeconds": demo_time_ticks / LEGACY_TICKS_PER_SECOND if pd.notna(demo_time_ticks) else pd.NA,
            "demoActionCount": action_count,
            "godMode": bool(demo.get("godMode")),
            "player": demo.get("player"),
            "model": solver.get("model"),
            "modelProfile": solver.get("modelProfile"),
            "provider": solver.get("provider"),
        })
    return pd.DataFrame(rows)


def build_runs_df(runs: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for trace_id, run in runs.items():
        model = run.get("model") or {}
        config = run.get("config") or {}
        created = iso_to_datetime(run.get("createdAt"))
        updated = iso_to_datetime(run.get("updatedAt"))
        duration = (updated - created).total_seconds() if pd.notna(created) and pd.notna(updated) else pd.NA
        rows.append({
            "traceId": run.get("id") or trace_id,
            "playData": run.get("playData"),
            "level": run.get("level"),
            "createdAt": created,
            "updatedAt": updated,
            "wallDurationSeconds": duration,
            "stepCount": run.get("stepCount"),
            "latestKeyCode": nested_get(run, "latestAction", "keyCode"),
            "latestTicks": nested_get(run, "latestAction", "ticks"),
            "model": model.get("model"),
            "modelProfile": model.get("modelProfile"),
            "provider": model.get("provider"),
            "modelSource": model.get("modelSource"),
            "candidateLimit": config.get("candidateLimit"),
            "maxActionTicks": config.get("maxActionTicks"),
            "temperature": config.get("temperature"),
            "showCandidateScores": config.get("showCandidateScores"),
        })
    return pd.DataFrame(rows)


def build_steps_df(runs: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for trace_id, run in runs.items():
        for index, step in enumerate(run.get("steps") or []):
            state = step.get("state") or {}
            runner = state.get("runner") or {}
            gold = state.get("gold") or {}
            guard_risk = state.get("guardRisk") or {}
            validation = step.get("validation") or {}
            stall = step.get("stallSupervisor") or {}
            action = step.get("action") or {}
            rows.append({
                "traceId": trace_id,
                "stepIndex": step.get("stepIndex", index),
                "createdAt": iso_to_datetime(step.get("createdAt")),
                "tick": state.get("tick"),
                "gameState": state.get("gameState"),
                "godMode": state.get("godMode"),
                "runnerX": runner.get("x"),
                "runnerY": runner.get("y"),
                "runnerAction": runner.get("action"),
                "goldComplete": gold.get("complete"),
                "goldRemaining": gold.get("remainingCount"),
                "visibleGoldCount": list_len(gold.get("visiblePositions")),
                "guardRisk": guard_risk.get("risk"),
                "selectedCandidateId": step.get("selectedCandidateId"),
                "selectedCandidateKind": step.get("selectedCandidateKind"),
                "keyCode": action.get("keyCode"),
                "ticks": action.get("ticks"),
                "validationFallbackUsed": validation.get("fallbackUsed"),
                "validationFallbackReason": validation.get("fallbackReason"),
                "validationStallBlocked": validation.get("stallBlocked"),
                "stallSeverity": stall.get("severity"),
                "stallType": stall.get("type"),
                "stallFallbackUsed": stall.get("fallbackUsed"),
                "candidateCount": list_len(step.get("candidates")),
            })
    return pd.DataFrame(rows)


def build_candidates_df(runs: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for trace_id, run in runs.items():
        for index, step in enumerate(run.get("steps") or []):
            selected_id = step.get("selectedCandidateId")
            for rank, candidate in enumerate(step.get("candidates") or [], start=1):
                first_action = candidate.get("firstAction") or {}
                rows.append({
                    "traceId": trace_id,
                    "stepIndex": step.get("stepIndex", index),
                    "rank": rank,
                    "candidateId": candidate.get("id"),
                    "kind": candidate.get("kind"),
                    "score": candidate.get("score"),
                    "selected": candidate.get("id") == selected_id,
                    "stallBlocked": candidate.get("stallBlocked"),
                    "stallRecovery": candidate.get("stallRecovery"),
                    "keyCode": first_action.get("keyCode"),
                    "ticks": first_action.get("ticks"),
                    "goal": candidate.get("goal"),
                    "reason": candidate.get("reason"),
                    "target": candidate.get("target"),
                })
    return pd.DataFrame(rows)

records_df = build_records_df(records)
runs_df = build_runs_df(runs)
steps_df = build_steps_df(runs)
candidates_df = build_candidates_df(runs)

joined_runs_df = records_df.merge(runs_df, on="traceId", how="left", suffixes=("Record", "Run")) if not records_df.empty else pd.DataFrame()

for name, df in [
    ("records_df", records_df),
    ("runs_df", runs_df),
    ("steps_df", steps_df),
    ("candidates_df", candidates_df),
    ("joined_runs_df", joined_runs_df),
]:
    print(f"{name}: {df.shape}")


## Recording And Run Summary


In [ ]:
run_cols = [
    "traceId_short", "modelRun", "result", "stepCount", "wallDurationSeconds",
    "demoTimeSeconds", "godMode", "savedAt",
]
if joined_runs_df.empty:
    print("No trace runs found.")
else:
    joined_runs_df["traceId_short"] = (
        joined_runs_df["traceId"]
        .fillna("")
        .astype(str)
        .str.split("-", n=1)
        .str[0]
    )
    display(joined_runs_df[[col for col in run_cols if col in joined_runs_df.columns]].sort_values("savedAt"))


## Overview Charts


In [ ]:
def plot_bar(series: pd.Series, title: str, xlabel: str = "", ylabel: str = "count") -> None:
    series = series.dropna()
    if series.empty:
        print(f"No data for {title}")
        return
    ax = series.value_counts().sort_values().plot(kind="barh", figsize=(8, max(3, len(series.value_counts()) * 0.35)))
    ax.set_title(title)
    ax.set_xlabel(ylabel)
    ax.set_ylabel(xlabel)
    plt.tight_layout()
    plt.show()

if not records_df.empty:
    plot_bar(records_df["source"].astype(str) + " " + records_df["result"].astype(str), "Recording outcomes")
    records_df.sort_values("savedAt").plot(x="savedAt", y="demoTimeSeconds", kind="bar", figsize=(9, 3), title="Demo time by recording")
    plt.ylabel("seconds")
    plt.tight_layout()
    plt.show()

if not runs_df.empty:
    plot_bar(runs_df["model"], "Model usage")
    runs_df.sort_values("createdAt").plot(x="createdAt", y="stepCount", kind="bar", figsize=(9, 3), title="Trace step count by run")
    plt.ylabel("steps")
    plt.tight_layout()
    plt.show()


## Candidate And Stall Analytics


In [ ]:
if steps_df.empty:
    print("No trace steps found.")
else:
    plot_bar(steps_df["selectedCandidateKind"], "Selected candidate kinds")
    plot_bar(steps_df["stallType"], "Stall types")
    display(
        steps_df.groupby("traceId", dropna=False)
        .agg(
            steps=("stepIndex", "count"),
            fallbacks=("validationFallbackUsed", lambda s: int(s.fillna(False).sum())),
            stall_blocks=("validationStallBlocked", lambda s: int(s.fillna(False).sum())),
            stalled_steps=("stallSeverity", lambda s: int((s == "stalled").sum())),
            first_tick=("tick", "min"),
            last_tick=("tick", "max"),
        )
        .reset_index()
    )

if candidates_df.empty:
    print("No candidate rows found.")
else:
    plot_bar(candidates_df["kind"], "Offered candidate kinds")
    display(
        candidates_df.groupby("kind", dropna=False)
        .agg(
            offered=("candidateId", "count"),
            selected=("selected", lambda s: int(s.fillna(False).sum())),
            avg_score=("score", "mean"),
            blocked=("stallBlocked", lambda s: int(s.fillna(False).sum())),
            recovery=("stallRecovery", lambda s: int(s.fillna(False).sum())),
        )
        .sort_values(["selected", "offered"], ascending=False)
        .reset_index()
    )


## Trace Drill-Down

Set `TRACE_ID` to inspect one retained run. By default, this picks the newest trace run if one exists.


In [ ]:
TRACE_ID = None

if TRACE_ID is None and not runs_df.empty:
    TRACE_ID = runs_df.sort_values("updatedAt", ascending=False).iloc[0]["traceId"]

print(f"TRACE_ID = {TRACE_ID}")

if TRACE_ID is None:
    print("No trace available.")
else:
    run_steps = steps_df[steps_df["traceId"] == TRACE_ID].copy() if not steps_df.empty else pd.DataFrame()
    run_candidates = candidates_df[candidates_df["traceId"] == TRACE_ID].copy() if not candidates_df.empty else pd.DataFrame()

    if run_steps.empty:
        print("No steps for selected trace.")
    else:
        display(run_steps[[
            "stepIndex", "tick", "runnerX", "runnerY", "goldRemaining", "guardRisk",
            "selectedCandidateKind", "selectedCandidateId", "keyCode", "ticks",
            "stallSeverity", "stallType", "validationFallbackUsed",
        ]])

    if run_candidates.empty:
        print("No candidates for selected trace.")
    else:
        selected = run_candidates[run_candidates["selected"]]
        display(selected[["stepIndex", "rank", "kind", "candidateId", "score", "goal", "reason"]])


## Raw DataFrames

Use these for ad hoc analysis:

- `records_df`
- `runs_df`
- `steps_df`
- `candidates_df`
- `joined_runs_df`
